In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
#%pip install pyarrow
#%pip install openpyxl


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   -- ------------------------------------- 1.8/28.6 MB 16.7 MB/s eta 0:00:02
   ----------- ---------------------------- 8.4/28.6 MB 26.8 MB/s eta 0:00:01
   -------------------------------- ------- 23.1/28.6 MB 44.8 MB/s eta 0:00:01
   ---------------------------------------- 28.6/28.6 MB 41.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
DATA_DIR = Path("C:\\Users\\Home\\Documents\\Datos Ebsa")
OUT_DIR = Path("C:\\Users\\Home\\Documents\\Datos Ebsa\\Procesado")

OUT_DIR.mkdir(exist_ok=True)

archivos = sorted(DATA_DIR.glob("*.xlsx"))

print(f"Archivos encontrados: {len(archivos)}")

for archivo in archivos[:10]:
    print(archivo.name)

Archivos encontrados: 12
formato_tc2_20251.xlsx
formato_tc2_202510.xlsx
formato_tc2_202511.xlsx
formato_tc2_202512.xlsx
formato_tc2_20252.xlsx
formato_tc2_20253.xlsx
formato_tc2_20254.xlsx
formato_tc2_20255.xlsx
formato_tc2_20256.xlsx
formato_tc2_20257.xlsx


In [9]:
COLUMNAS = [
    "NIU",
    "ID Factura",
    "Tipo Factura",
    "Año de reporte",
    "Mes de reporte",
    "Días Facturados",
    "Consumo Usuario (kWh)",
    "Refacturación por Consumo Usuario - (kWh)",
    "Fecha de Lectura Actual",
    "Fecha de Lectura Anterior",
    "Tipo de Lectura",
    "Ciclo",
    "Clase de Servicio"
]

In [10]:
def procesar_archivo(ruta):
    
    print(f"Procesando: {ruta.name}")
    
    df = pd.read_excel(
        ruta,
        usecols=lambda col: col in COLUMNAS,
        engine="openpyxl"
    )

    # -------------------------
    # NIU
    # -------------------------
    
    df["NIU"] = (
        df["NIU"]
        .astype("string")
        .str.strip()
    )

    # -------------------------
    # Variables numéricas
    # -------------------------
    
    numericas = [
        "Año de reporte",
        "Mes de reporte",
        "Días Facturados",
        "Consumo Usuario (kWh)",
        "Refacturación por Consumo Usuario - (kWh)",
        "Ciclo"
    ]

    for col in numericas:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # -------------------------
    # Fechas
    # -------------------------

    df["Fecha de Lectura Actual"] = pd.to_datetime(
        df["Fecha de Lectura Actual"],
        errors="coerce",
        dayfirst=True
    )

    df["Fecha de Lectura Anterior"] = pd.to_datetime(
        df["Fecha de Lectura Anterior"],
        errors="coerce",
        dayfirst=True
    )

    # -------------------------
    # Eliminar registros sin
    # información básica
    # -------------------------

    df = df.dropna(
        subset=[
            "NIU",
            "Año de reporte",
            "Mes de reporte"
        ]
    )

    df["Año de reporte"] = df["Año de reporte"].astype(int)
    df["Mes de reporte"] = df["Mes de reporte"].astype(int)

    # -------------------------
    # Crear fecha mensual
    # -------------------------

    df["periodo"] = pd.to_datetime(
        dict(
            year=df["Año de reporte"],
            month=df["Mes de reporte"],
            day=1
        )
    )

    return df

In [11]:
resumenes = []

for archivo in archivos:
    
    df = procesar_archivo(archivo)

    resumen = (
        df
        .groupby(
            ["NIU", "periodo"],
            as_index=False
        )
        .agg(
            cantidad_registros=("ID Factura", "size"),

            dias_facturados_max=(
                "Días Facturados",
                "max"
            ),

            dias_facturados_mediana=(
                "Días Facturados",
                "median"
            ),

            fecha_lectura_actual=(
                "Fecha de Lectura Actual",
                "max"
            ),

            fecha_lectura_anterior=(
                "Fecha de Lectura Anterior",
                "min"
            ),

            ciclo=(
                "Ciclo",
                "first"
            ),

            ciclos_diferentes=(
                "Ciclo",
                "nunique"
            ),

            tipo_lectura=(
                "Tipo de Lectura",
                "first"
            ),

            clase_servicio=(
                "Clase de Servicio",
                "first"
            )
        )
    )

    resumenes.append(resumen)

    # Liberar memoria
    del df

Procesando: formato_tc2_20251.xlsx
Procesando: formato_tc2_202510.xlsx
Procesando: formato_tc2_202511.xlsx
Procesando: formato_tc2_202512.xlsx
Procesando: formato_tc2_20252.xlsx
Procesando: formato_tc2_20253.xlsx
Procesando: formato_tc2_20254.xlsx
Procesando: formato_tc2_20255.xlsx
Procesando: formato_tc2_20256.xlsx
Procesando: formato_tc2_20257.xlsx
Procesando: formato_tc2_20258.xlsx
Procesando: formato_tc2_20259.xlsx


In [12]:
historico = pd.concat(
    resumenes,
    ignore_index=True
)

In [13]:
historico.shape

(5119427, 11)

In [18]:
historico.to_pickle(
    OUT_DIR / "historico_temporal.pkl"
)

print("Histórico temporal guardado")

Histórico temporal guardado


In [19]:
historico["presente"] = 1

In [20]:
matriz_presencia = (
    historico
    .pivot_table(
        index="NIU",
        columns="periodo",
        values="presente",
        aggfunc="max",
        fill_value=0
    )
    .astype("uint8")
)

In [21]:
meses = pd.date_range(
    start="2022-01-01",
    end="2025-12-01",
    freq="MS"
)

In [22]:
matriz_presencia = matriz_presencia.reindex(
    columns=meses,
    fill_value=0
)

In [23]:
matriz_presencia.shape

(579632, 48)

In [25]:
conteo_por_mes = (
    matriz_presencia
    .sum(axis=0)
    .rename("clientes_presentes")
    .to_frame()
)

conteo_por_mes

,clientes_presentes
2022-01-01,0.0
2022-02-01,0.0
2022-03-01,0.0
2022-04-01,0.0
2022-05-01,0.0
2022-06-01,0.0
2022-07-01,0.0
2022-08-01,0.0
2022-09-01,0.0
2022-10-01,0.0


In [26]:
conteo_por_mes[
    conteo_por_mes["clientes_presentes"] > 0
]

,clientes_presentes
2025-01-01,563316.0
2025-02-01,351570.0
2025-03-01,352071.0
2025-04-01,567052.0
2025-05-01,353761.0
2025-06-01,354575.0
2025-07-01,570833.0
2025-08-01,356302.0
2025-09-01,357094.0
2025-10-01,574727.0


In [27]:
meses_2025 = pd.date_range(
    start="2025-01-01",
    end="2025-12-01",
    freq="MS"
)

matriz_2025 = matriz_presencia.reindex(
    columns=meses_2025,
    fill_value=0
)

matriz_2025.shape

(579632, 12)

In [ ]:
#numero de veces que aparecen los clientes por mes
meses_por_cliente = matriz_2025.sum(axis=1)

meses_por_cliente.value_counts().sort_index()

1       5427
2       2509
3       2185
4     213273
5       1033
6        904
7        864
8        791
9       1011
10       659
11      1031
12    349945
Name: count, dtype: int64

In [30]:
resumen_apariciones = (
    meses_por_cliente
    .value_counts()
    .sort_index()
    .rename_axis("meses_presentes")
    .reset_index(name="cantidad_clientes")
)

resumen_apariciones

,meses_presentes,cantidad_clientes
0,1,5427
1,2,2509
2,3,2185
3,4,213273
4,5,1033
5,6,904
6,7,864
7,8,791
8,9,1011
9,10,659


In [31]:
resumen_apariciones["porcentaje"] = (
    resumen_apariciones["cantidad_clientes"]
    / len(matriz_2025)
    * 100
)

resumen_apariciones

,meses_presentes,cantidad_clientes,porcentaje
0,1,5427,0.936284
1,2,2509,0.432861
2,3,2185,0.376963
3,4,213273,36.794552
4,5,1033,0.178217
5,6,904,0.155961
6,7,864,0.149060
7,8,791,0.136466
8,9,1011,0.174421
9,10,659,0.113693


In [32]:
clientes_4_meses = matriz_2025[
    matriz_2025.sum(axis=1) == 4
]

clientes_4_meses.shape

(213273, 12)

In [33]:
clientes_4_meses.head(20)

,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01
NIU,,,,,,,,,,,,
100000949,1,0,0,1,0,0,1,0,0,1,0,0
100001726,1,0,0,1,0,0,1,0,0,1,0,0
100002503,1,0,0,1,0,0,1,0,0,1,0,0
100003390,1,0,0,1,0,0,1,0,0,1,0,0
100004177,1,0,0,1,0,0,1,0,0,1,0,0
100005854,1,0,0,1,0,0,1,0,0,1,0,0
100006631,1,0,0,1,0,0,1,0,0,1,0,0
100007418,1,0,0,1,0,0,1,0,0,1,0,0
100008205,1,0,0,1,0,0,1,0,0,1,0,0


In [34]:
import numpy as np

def intervalos_aparicion(fila):
    
    posiciones = np.flatnonzero(
        fila.to_numpy() == 1
    )
    
    if len(posiciones) < 2:
        return []
    
    return np.diff(posiciones).tolist()

In [35]:
ejemplo_intervalos = clientes_4_meses.apply(
    intervalos_aparicion,
    axis=1
)

ejemplo_intervalos.head(20)

NIU
100000949     [3, 3, 3]
100001726     [3, 3, 3]
100002503     [3, 3, 3]
100003390     [3, 3, 3]
100004177     [3, 3, 3]
100005854     [3, 3, 3]
100006631     [3, 3, 3]
100007418     [3, 3, 3]
100008205     [3, 3, 3]
100009082     [3, 3, 3]
100010818     [3, 3, 3]
100011605     [3, 3, 3]
100012482     [3, 3, 3]
1000127971    [3, 3, 3]
1000128758    [3, 3, 3]
1000129535    [3, 3, 3]
1000130371    [3, 3, 3]
100013269     [3, 3, 3]
100014046     [3, 3, 3]
100015723     [3, 3, 3]
dtype: object

In [36]:
def es_patron_trimestral(fila):
    
    posiciones = np.flatnonzero(
        fila.to_numpy() == 1
    )
    
    if len(posiciones) < 3:
        return False
    
    diferencias = np.diff(posiciones)
    
    return np.all(
        (diferencias >= 2) &
        (diferencias <= 4)
    )

In [37]:
patron_trimestral = matriz_2025.apply(
    es_patron_trimestral,
    axis=1
)

In [38]:
patron_trimestral.value_counts()

False    365871
True     213761
Name: count, dtype: int64

In [39]:
nius_patron_trimestral = matriz_2025.index[
    patron_trimestral
]

len(nius_patron_trimestral)

213761

In [40]:
candidatos = historico[
    historico["NIU"].isin(
        nius_patron_trimestral
    )
].copy()

In [41]:
candidatos[
    "dias_facturados_max"
].describe(
    percentiles=[
        .25,
        .50,
        .75,
        .90,
        .95,
        .99
    ]
)

count    853714.000000
mean         91.126713
std           5.119046
min           0.000000
25%          89.000000
50%          91.000000
75%          94.000000
90%          96.000000
95%          99.000000
99%         101.000000
max         129.000000
Name: dias_facturados_max, dtype: float64

In [50]:
candidatos["lectura_90_dias"] = (
    candidatos["dias_facturados_max"]
    .between(80, 130)
)

candidatos[
    "lectura_90_dias"
].value_counts(normalize=True)

lectura_90_dias
True     0.984086
False    0.015914
Name: proportion, dtype: float64

In [51]:
conteo_por_mes[
    conteo_por_mes["clientes_presentes"] > 0
]

,clientes_presentes
2025-01-01,563316.0
2025-02-01,351570.0
2025-03-01,352071.0
2025-04-01,567052.0
2025-05-01,353761.0
2025-06-01,354575.0
2025-07-01,570833.0
2025-08-01,356302.0
2025-09-01,357094.0
2025-10-01,574727.0


In [52]:
resumen_apariciones

,meses_presentes,cantidad_clientes,porcentaje
0,1,5427,0.936284
1,2,2509,0.432861
2,3,2185,0.376963
3,4,213273,36.794552
4,5,1033,0.178217
5,6,904,0.155961
6,7,864,0.149060
7,8,791,0.136466
8,9,1011,0.174421
9,10,659,0.113693
